[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Your First Test &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the project's folder and `run_pytest`, `pytest_report` and `exit_code`, and
the three cells after it write the module and the test files as the notebook's worked examples left
them. Run them first, then the tasks in order, since tasks 2 to 5 use the file task 1 writes. The
last cell removes the scratch folder.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
PROJECT.mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


def exit_code(*arguments):
    """The exit code of python -m pytest, run in the project's folder with these arguments."""
    return subprocess.run([sys.executable, "-m", "pytest", *arguments], cwd=PROJECT, capture_output=True).returncode


print("ready:", PROJECT)


ready: scratch/stations


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


In [3]:
%%writefile scratch/stations/test_readings.py
from readings import mean, parse_reading, to_fahrenheit


def test_mean_of_two_readings():
    assert mean([4.2, 5.8]) == 5.0


def test_mean_skips_missing_readings():
    assert mean([4.2, None, 5.8]) == 5.0


def test_mean_of_no_readings_is_none():
    assert mean([None, None]) is None


def test_an_empty_reading_is_none():
    assert parse_reading("Svalbard,") == ("Svalbard", None)


def test_freezing_point_in_fahrenheit():
    assert to_fahrenheit(0) == 32


def test_boiling_point_in_fahrenheit():
    assert to_fahrenheit(100) == 212


Writing scratch/stations/test_readings.py


In [4]:
%%writefile scratch/stations/test_summary.py
from readings import summarize


def tuesday():
    """Tuesday's lines of readings, for the tests below."""
    return ["Bergen,4.2", "Bergen,5.8", "Bergen,", "Oslo,-2.4", "Oslo,-1.6",
            "Svalbard,", "Svalbard,", "Tromso,-6.3", "Tromso,-5.7", "Tromso,"]


def test_summary_of_tuesday():
    assert summarize(tuesday()) == {"Bergen": 5.0, "Oslo": -2.0, "Svalbard": None, "Tromso": -6.0}


def test_every_station_is_in_the_summary():
    stations = sorted(summarize(tuesday()))
    assert stations == ["Bergen", "Oslo", "Svalbard", "Tromso"], "a station is missing from the summary"


def test_a_blank_line_is_skipped():
    assert summarize(["Bergen,4.2", "", "Bergen,5.8"]) == {"Bergen": 5.0}


Writing scratch/stations/test_summary.py


**1.** A test file of your own, run alone.


In [5]:
%%writefile scratch/stations/test_tasks.py
from readings import mean


def test_mean_of_one_reading():
    assert mean([-6.3]) == -6.3


Writing scratch/stations/test_tasks.py


In [6]:
run_pytest("test_tasks.py")


============================= test session starts ==============================
collected 1 item

test_tasks.py .                                                          [100%]

============================== 1 passed ===============================


A path after `pytest` runs that file and nothing else, so `collected 1 item` counts only its test.


**2.** A test that fails on purpose.


In [7]:
%%writefile scratch/stations/test_tasks.py
from readings import mean


def test_mean_of_one_reading():
    assert mean([-6.3]) == -6.3


def test_mean_is_the_larger_reading():
    assert mean([4.2, 5.8]) == 5.8


Overwriting scratch/stations/test_tasks.py


In [8]:
run_pytest("test_tasks.py")


============================= test session starts ==============================
collected 2 items

test_tasks.py .F                                                         [100%]

=================================== FAILURES ===================================
_______________________ test_mean_is_the_larger_reading ________________________

    def test_mean_is_the_larger_reading():
>       assert mean([4.2, 5.8]) == 5.8
E       assert 5.0 == 5.8
E        +  where 5.0 = mean([4.2, 5.8])

test_tasks.py:9: AssertionError
=========================== short test summary info ============================
FAILED test_tasks.py::test_mean_is_the_larger_reading - assert 5.0 == 5.8
========================= 1 failed, 1 passed ==========================


The value is in the `E` line: `assert 5.0 == 5.8`, with `where 5.0 = mean([4.2, 5.8])` under it.


**3.** One test, by its node ID.


In [9]:
run_pytest("test_tasks.py::test_mean_is_the_larger_reading", "-v")


============================= test session starts ==============================
collecting ... collected 1 item

test_tasks.py::test_mean_is_the_larger_reading FAILED                    [100%]

=================================== FAILURES ===================================
_______________________ test_mean_is_the_larger_reading ________________________

    def test_mean_is_the_larger_reading():
>       assert mean([4.2, 5.8]) == 5.8
E       assert 5.0 == 5.8
E        +  where 5.0 = mean([4.2, 5.8])

test_tasks.py:9: AssertionError
=========================== short test summary info ============================
FAILED test_tasks.py::test_mean_is_the_larger_reading - assert 5.0 == 5.8
============================== 1 failed ===============================


The node ID is the file and the test's name joined by `::`, exactly as the short test summary printed
it in task 2.


**4.** Tests picked by a keyword.


In [10]:
run_pytest("-k", "fahrenheit", "-v")


============================= test session starts ==============================
collecting ... collected 11 items / 9 deselected / 2 selected

test_readings.py::test_freezing_point_in_fahrenheit PASSED               [ 50%]
test_readings.py::test_boiling_point_in_fahrenheit PASSED                [100%]

======================= 2 passed, 9 deselected ========================


The other tests were collected and deselected, and the counts on the last line say how many.


**5.** The exit code, before and after the correction.


In [11]:
print("before:", exit_code("test_tasks.py"))

(PROJECT / "test_tasks.py").write_text((PROJECT / "test_tasks.py").read_text().replace("== 5.8", "== 5.0"))

print("after: ", exit_code("test_tasks.py"))


before: 1
after:  0


1 while a test failed, and 0 once every test passed. `replace` changed the one expected value in the
file, which a `%%writefile` of the whole file would also have done.


**6.** Every test in the project, counted.


In [12]:
report = pytest_report("--collect-only", "-q")

print(report)
print("tests:", sum("::" in line for line in report.splitlines()))


test_readings.py::test_mean_of_two_readings
test_readings.py::test_mean_skips_missing_readings
test_readings.py::test_mean_of_no_readings_is_none
test_readings.py::test_an_empty_reading_is_none
test_readings.py::test_freezing_point_in_fahrenheit
test_readings.py::test_boiling_point_in_fahrenheit
test_summary.py::test_summary_of_tuesday
test_summary.py::test_every_station_is_in_the_summary
test_summary.py::test_a_blank_line_is_skipped
test_tasks.py::test_mean_of_one_reading
test_tasks.py::test_mean_is_the_larger_reading

11 tests collected
tests: 11


The count agrees with the line pytest prints under the list. Counting the lines with `::` is the kind
of thing a script does with a report, when it needs a number rather than a person's reading.

Last, remove the scratch folder:


In [13]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Your First Test](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/02-your-first-test.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
